# NLP Feature Setup: 10-K AI-disclosure front end

Shared front end for the three AI-maturity dimensions (strategy,
operations, governance). It resolves the latest 10-K per Fortune 500
firm, extracts Items 1, 1A, and 7, segments and AI-filters their
sentences, and validates the AI keyword dictionary. The cached outputs
(`filings`, `sentences`, `sentence_totals`, `keyword_distribution`) are
consumed by the per-dimension notebooks, which handle tone scoring and
firm-level aggregation.

Run this notebook once before `strategy.ipynb`, `operations.ipynb`, or
`governance.ipynb`. It is the only notebook that runs the spaCy pass.

**Prerequisites.**
- The conda environment from `environment.yml` is active (provides
  `spacy` + `en_core_web_sm`, `edgartools`, `scikit-learn`).
- `EDGAR_IDENTITY` set to "Your Name email@host" (the setup cell falls
  back to a default if unset).

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import logging
import os
import random
import sys
import webbrowser
from collections import Counter
from pathlib import Path

import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.indicators.common.io import (
    cache_path,
    clear_cache,
    load_cached_step,
    save_cached_step,
)
from src.indicators.nlp_features import (
    AI_KEYWORDS,
    assemble_sections,
    build_ai_sentence_review,
    extract_items,
    fetch_filing,
    filter_ai_sentences,
    resolve_fortune500_filings,
)
from src.indicators.nlp_features.filter import (
    aggregate_keyword_counts,
    count_keyword_occurrences,
    is_ai_sentence,
    split_sentences,
)
from src.indicators.nlp_features.aggregate import ALL_ITEMS, MIN_ITEM_SENTENCES_FOR_PARSE

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

os.environ.setdefault("EDGAR_IDENTITY", "Timo Koba kab.timo3@gmail.com")

# Cached steps under data_cache/indicators/nlp_features/. Only `edgar_sections`
# and `raw_text` (section 2a) touch EDGAR; everything else is rebuilt offline
# from them, so re-tuning the parser or splitter never re-downloads:
#   filings          - resolved 10-K per ticker (section 1)
#   edgar_sections   - edgartools structured Item picks    (2a, EDGAR)
#   raw_text         - full filing text for the regex path (2a, EDGAR)
#   sections         - final Items + parser provenance     (2b, local)
#   sentences / sentence_totals / keyword_distribution      (2b, local)
# Toggle FORCE_REFRESH to recompute all, or clear_cache("nlp_features", "<step>").
FORCE_REFRESH = False
SHARED = "nlp_features"

# Validation-sample knobs (section 3). Surfaced here so a reviewer can find
# them without scrolling into the validation cells.
VALIDATION_DIR = PROJECT_ROOT / "data_clean" / "validation" / SHARED
ANNOTATION_FILE = VALIDATION_DIR / "sample_to_annotate.csv"
KEY_FILE = VALIDATION_DIR / "_key.parquet"
SAMPLE_FILINGS_N = 30
POS_N = 100
NEG_N = 100
SEED = 42

## 1. Resolve filings

Look up the latest 10-K for each Fortune 500 ticker. Foreign filers
(20-F) and resolution failures are dropped and logged.

In [ ]:
filings = None if FORCE_REFRESH else load_cached_step(SHARED, "filings")
if filings is None:
    filings = resolve_fortune500_filings(fiscal_year=None)
    save_cached_step(filings, SHARED, "filings")
    print(f"Resolved {len(filings)} 10-K filings (saved to {cache_path(SHARED, 'filings')})")
else:
    print(f"Loaded {len(filings)} 10-K filings from cache ({cache_path(SHARED, 'filings')})")
filings.head()

## 2. Parse Item sections + filter AI sentences

Two stages so the EDGAR round-trip happens once and everything else stays local:

- **2a — Download (EDGAR).** For each filing, fetch edgartools' structured Item
  picks and, only when an Item is missing, the full raw text. Cached as
  `edgar_sections` and `raw_text`.
- **2b — Assemble + filter (offline).** Combine the edgartools picks with a
  line-anchored **regex fallback** over the raw text for any missing Item; every
  candidate (structured or regex) passes the same validation, and each returned
  Item is tagged in `sections` with the parser that produced it
  (`edgartools` / `regex`). Then sentence-segment and split each Item's sentences
  into AI-relevant (matched against `AI_KEYWORDS`) plus total cleaned counts. The
  cleaned-sentence totals are the denominator of `ai_sentence_share` in each
  dimension notebook (length-normalized AI-disclosure intensity, Loughran-McDonald
  2011). `sentences` keeps an `item` column so each dimension subsets to its Item.

Re-tuning the regex or the splitter means deleting the `sections` / `sentences` /
`sentence_totals` / `keyword_distribution` steps and re-running 2b — no EDGAR.

**Parse coverage and the `parse_complete` flag.** Regex-sourced sections are
lower-confidence (they can truncate or over-capture), so they are flagged red in
the section-4 review for manual audit. A residual set of large financial firms
(e.g. JPMorgan, MetLife, KKR) cannot be recovered from the 10-K at all because
they *incorporate Items 7, 7A, and 8 by reference from Exhibit 13*, so their MD&A
is not in the primary document; regex cannot help there either. A filing counts
as parsed for an Item only when that Item yields at least
`MIN_ITEM_SENTENCES_FOR_PARSE` clean sentences; `aggregate.py` writes an identical
`parse_complete` flag into every dimension's output, so incomplete firms are
dropped uniformly across strategy, operations, and governance rather than each
dimension silently losing a different subset.

In [ ]:
# 2a. Download & cache each filing's edgartools sections + full raw text.
# This is the ONLY EDGAR-bound step. Raw text is cached for every filing so the
# regex fallback can later be re-run (and re-tuned) against any of them offline.
print(f"AI_KEYWORDS: {len(AI_KEYWORDS)} keywords")

edgar_sections_df = None if FORCE_REFRESH else load_cached_step(SHARED, "edgar_sections")
raw_text_df = None if FORCE_REFRESH else load_cached_step(SHARED, "raw_text")

if edgar_sections_df is None or raw_text_df is None:
    sec_rows: list[dict] = []
    raw_rows: list[dict] = []
    fetch_errors: list[tuple[str, str]] = []
    for _, row in filings.iterrows():
        acc = row["accession_number"]
        try:
            fetched = fetch_filing(acc)
            for item, text in fetched.edgar_sections.items():
                sec_rows.append({"accession_number": acc, "cik": row["cik"], "item": item, "text": text})
            if fetched.raw_text:
                raw_rows.append({"accession_number": acc, "cik": row["cik"], "text": fetched.raw_text})
        except Exception as exc:
            fetch_errors.append((acc, str(exc)))

    edgar_sections_df = pd.DataFrame(sec_rows)
    raw_text_df = pd.DataFrame(raw_rows)
    save_cached_step(edgar_sections_df, SHARED, "edgar_sections")
    save_cached_step(raw_text_df, SHARED, "raw_text")
    print(f"Fetched edgartools sections for {edgar_sections_df['accession_number'].nunique()} filings "
          f"({len(edgar_sections_df)} Item rows) and raw text for {len(raw_rows)} filings. "
          f"Fetch errors: {len(fetch_errors)}")
else:
    print(f"Loaded cached edgartools sections ({len(edgar_sections_df)} Item rows) and raw text "
          f"({len(raw_text_df)} filings) — no EDGAR calls.")

In [ ]:
# 2b. Assemble final sections (edgartools + validated regex fallback, with
# per-Item provenance), then segment + AI-filter. Fully offline: rebuilt from the
# 2a caches, so this is what you re-run after changing parse.py or filter.py.
sections_df = None if FORCE_REFRESH else load_cached_step(SHARED, "sections")
sentences_df = None if FORCE_REFRESH else load_cached_step(SHARED, "sentences")
sentence_totals_df = None if FORCE_REFRESH else load_cached_step(SHARED, "sentence_totals")
keyword_distribution_df = None if FORCE_REFRESH else load_cached_step(SHARED, "keyword_distribution")

if any(x is None for x in (sections_df, sentences_df, sentence_totals_df, keyword_distribution_df)):
    edgar_by_acc = {
        acc: dict(zip(g["item"], g["text"])) for acc, g in edgar_sections_df.groupby("accession_number")
    }
    raw_by_acc = dict(zip(raw_text_df["accession_number"], raw_text_df["text"])) if len(raw_text_df) else {}

    sec_rows: list[dict] = []
    all_sentences: list[pd.DataFrame] = []
    totals_rows: list[dict] = []
    for _, row in filings.iterrows():
        acc = row["accession_number"]
        sections, sources = assemble_sections(edgar_by_acc.get(acc, {}), raw_by_acc.get(acc, ""))
        for item, text in sections.items():
            sec_rows.append(
                {"accession_number": acc, "cik": row["cik"], "item": item, "source": sources[item], "text": text}
            )
        sub, totals = filter_ai_sentences(sections, accession_number=acc, cik=row["cik"])
        if len(sub) > 0:
            all_sentences.append(sub)
        totals_rows.append(
            {
                "accession_number": acc,
                "cik": row["cik"],
                "n_sentences_item_1": int(totals.get("item_1", 0)),
                "n_sentences_item_1a": int(totals.get("item_1a", 0)),
                "n_sentences_item_7": int(totals.get("item_7", 0)),
            }
        )

    sections_df = pd.DataFrame(sec_rows)
    save_cached_step(sections_df, SHARED, "sections")

    sentences_df = pd.concat(all_sentences, ignore_index=True) if all_sentences else pd.DataFrame()
    save_cached_step(sentences_df, SHARED, "sentences")

    sentence_totals_df = pd.DataFrame(totals_rows)
    save_cached_step(sentence_totals_df, SHARED, "sentence_totals")

    # Concept frequencies over the analyzed AI sentences (every keyword occurrence
    # lies inside an AI sentence, so this is exact and needs no separate pass).
    kw_counter: Counter = Counter()
    for _sent in sentences_df["sentence"]:
        kw_counter.update(count_keyword_occurrences(_sent))
    keyword_distribution_df = aggregate_keyword_counts(kw_counter)
    save_cached_step(keyword_distribution_df, SHARED, "keyword_distribution")
    print(f"Assembled sections for {sections_df['accession_number'].nunique()} filings; "
          f"{len(sentences_df)} AI-relevant sentences.")
else:
    print(f"Loaded {len(sentences_df)} AI sentences from cache ({cache_path(SHARED, 'sentences')})")

_by_source = sections_df["source"].value_counts()
print(f"Sections by parser: edgartools={int(_by_source.get('edgartools', 0))}, "
      f"regex={int(_by_source.get('regex', 0))}  "
      f"(regex sections are flagged red in the section-4 review)")
print(f"Filings with at least one AI mention: "
      f"{sentences_df['accession_number'].nunique() if len(sentences_df) else 0}")

print(f"\nPer-Item parse coverage (>= {MIN_ITEM_SENTENCES_FOR_PARSE} clean sentences):")
_n_filings = len(sentence_totals_df)
for _item in ALL_ITEMS:
    _ok = int((sentence_totals_df[f"n_sentences_{_item}"] >= MIN_ITEM_SENTENCES_FOR_PARSE).sum())
    print(f"  {_item:8s}: {_ok:3d}/{_n_filings}")
_complete = (
    sentence_totals_df[[f"n_sentences_{i}" for i in ALL_ITEMS]]
    .ge(MIN_ITEM_SENTENCES_FOR_PARSE)
    .all(axis=1)
)
print(f"  parse_complete (all three Items): {int(_complete.sum())}/{_n_filings}  "
      f"(the {int((~_complete).sum())} incomplete filings are dropped uniformly via parse_complete)")

print("\nAI sentences by Item (extensive-margin raw counts):")
print(sentences_df["item"].value_counts().to_string() if len(sentences_df) else "(none)")
print(f"\nKeyword concept groups: {len(keyword_distribution_df)}  "
      f"(0-count: {int((keyword_distribution_df['count'] == 0).sum())})")
print("Top 15 concept groups by total occurrences:")
print(keyword_distribution_df.head(15).to_string(index=False))

## 3. Validation: keyword dictionary precision / recall / F1 *(optional, for appendix)*

Stratified random sample of 200 sentences (100 dict-positive, 100
dict-negative) drawn from a 30-filing subsample. Sentences are shuffled
and the source label is hidden so the annotator labels blind.

**Workflow.**
1. Run the **Build sample** cell once. It writes
   `data_clean/validation/nlp_features/sample_to_annotate.csv`.
2. Open that CSV, fill the `gold` column with `1` if the sentence
   substantively discusses AI/ML technology, deployment, governance, or
   strategy, else `0`. Save (keep the same filename).
3. Run the **Compute metrics** cell. It joins your annotations with the
   hidden source key and prints precision, recall, F1 plus example errors.

To regenerate the sample, delete the CSV.

In [ ]:
# Self-load `filings` from cache if it isn't already in memory (so this cell
# can run after a kernel restart without re-executing section 1).
try:
    filings
except NameError:
    filings = load_cached_step(SHARED, "filings")
    if filings is None:
        raise RuntimeError("No cached filings found. Run section 1 (Resolve filings) first.")
    print(f"Loaded {len(filings)} filings from cache for validation")

if ANNOTATION_FILE.exists():
    print(f"Sample already exists: {ANNOTATION_FILE}")
    print("Delete the file to regenerate, or annotate it and run the metrics cell.")
else:
    pool_filings = filings.sample(n=min(SAMPLE_FILINGS_N, len(filings)), random_state=SEED)
    pool_pos: list[dict] = []
    pool_neg: list[dict] = []
    for _, row in pool_filings.iterrows():
        try:
            sections = extract_items(row["accession_number"])
        except Exception:
            continue
        for item, text in sections.sections.items():
            for sent in split_sentences(text):
                rec = {"sentence": sent, "item": item, "accession_number": row["accession_number"]}
                (pool_pos if is_ai_sentence(sent) else pool_neg).append(rec)

    rng = random.Random(SEED)
    rng.shuffle(pool_pos)
    rng.shuffle(pool_neg)
    selected = pd.DataFrame(
        [{**r, "source": "pos"} for r in pool_pos[:POS_N]]
        + [{**r, "source": "neg"} for r in pool_neg[:NEG_N]]
    )
    selected = selected.sample(frac=1, random_state=SEED).reset_index(drop=True)
    selected.insert(0, "id", range(len(selected)))

    VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
    selected[["id", "source"]].to_parquet(KEY_FILE, index=False)
    annotation = selected[["id", "sentence", "item", "accession_number"]].copy()
    annotation["gold"] = ""
    annotation.to_csv(ANNOTATION_FILE, index=False, encoding="utf-8-sig")

    print(f"Pool sizes: {len(pool_pos)} positives / {len(pool_neg)} negatives across {len(pool_filings)} filings")
    print(f"Wrote {len(annotation)} sentences to: {ANNOTATION_FILE}")
    print("Open it, fill the 'gold' column with 1 or 0, save, then run the next cell.")

In [ ]:
if not ANNOTATION_FILE.exists() or not KEY_FILE.exists():
    raise RuntimeError(
        f"Validation files not found under {VALIDATION_DIR}. Run the Build sample cell first."
    )

annotated = pd.read_csv(ANNOTATION_FILE, encoding="utf-8-sig")
key = pd.read_parquet(KEY_FILE)
df = annotated.merge(key, on="id", how="inner")

df["gold"] = pd.to_numeric(df["gold"], errors="coerce")
unannotated = int(df["gold"].isna().sum())
df = df.dropna(subset=["gold"]).copy()
df["gold"] = df["gold"].astype(int)
df["pred"] = (df["source"] == "pos").astype(int)

if unannotated > 0:
    print(f"WARNING: {unannotated} rows have no gold label and were skipped.")

P = precision_score(df["gold"], df["pred"], zero_division=0)
R = recall_score(df["gold"], df["pred"], zero_division=0)
F = f1_score(df["gold"], df["pred"], zero_division=0)
tn, fp, fn, tp = confusion_matrix(df["gold"], df["pred"], labels=[0, 1]).ravel()

print(f"N annotated:  {len(df)}")
print(f"Precision:    {P:.3f}   ({tp} TP / {tp + fp} predicted positives)")
print(f"Recall:       {R:.3f}   ({tp} TP / {tp + fn} actual positives)")
print(f"F1:           {F:.3f}")
print(f"Confusion:    TP={tp}  FP={fp}  TN={tn}  FN={fn}")

print("\n-- Up to 5 false positives (dict said AI, you said not) --")
for _, r in df[(df["pred"] == 1) & (df["gold"] == 0)].head(5).iterrows():
    print(f"  [{r['item']}] {r['sentence']}")
print("\n-- Up to 5 false negatives (you said AI, dict missed it) --")
for _, r in df[(df["pred"] == 0) & (df["gold"] == 1)].head(5).iterrows():
    print(f"  [{r['item']}] {r['sentence']}")

## 4. Manual parse review (HTML)

Build one self-contained HTML page listing every parse-complete firm and its
AI-relevant sentences per Item, in document order (runs of non-AI sentences are
elided with a count). Each Item is badged with the parser that produced it, and
**regex-sourced Items are flagged red** — review those, since the regex fallback
can truncate or over-capture where edgartools declined to parse. The page opens
automatically in your browser. It reads only the cached frames, so it is safe to
re-run any time (no EDGAR, no recompute).

In [ ]:
# Self-load from cache so this runs standalone after a kernel restart.
try:
    filings
except NameError:
    filings = load_cached_step(SHARED, "filings")

_sent = load_cached_step(SHARED, "sentences")
_tot = load_cached_step(SHARED, "sentence_totals")
_sec = load_cached_step(SHARED, "sections")
if any(x is None for x in (filings, _sent, _tot, _sec)):
    raise RuntimeError("Run section 2 first — sentences / sentence_totals / sections caches are missing.")

review_path = VALIDATION_DIR / "parse_review_ai_sentences.html"
build_ai_sentence_review(_sent, filings, _tot, _sec, review_path)
_n_regex = int((_sec["source"] == "regex").sum())
print(f"Wrote {review_path}")
print(f"{_n_regex} regex-parsed section(s) flagged red — please review those.")
webbrowser.open(review_path.resolve().as_uri())